In [0]:
import sys
import os

print("Python version:", sys.version)
print("Python executable:", sys.executable)
print("Current directory:", os.getcwd())
print("PySpark version:", __import__("pyspark").__version__)

In [0]:
import sys
import importlib.util
from pathlib import Path

# Locate the repository root
repo_root = Path.cwd().parent

# Add the project root to Python's import path
sys.path.insert(0, str(repo_root))

print("Project root:", repo_root)
print("Pytest installed:", importlib.util.find_spec("pytest") is not None)

# Test project module imports
from src.ingestion.jsonstat_ingestion import load_jsonstat
from src.transformations.bronze import build_bronze_dataframe
from src.transformations.silver import build_silver_dataframe
from src.transformations.gold import build_gold_tables
from src.quality.validation import run_quality_checks

print("All project modules imported successfully!")

In [0]:
%pip install pytest

In [0]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pytest

exit_code = pytest.main([
    "-v",
    "--tb=short",
    "-p", "no:cacheprovider",
    "--assert=plain",
    str(repo_root / "tests"),
])

print("Pytest exit code:", exit_code)

In [0]:
from pyspark.sql import SparkSession

print("Spark session type:")
print(type(spark))

print("\nActive session:")
session = SparkSession.getActiveSession()
print(type(session))

print("\nSpark Connect:")
print(session.__class__.__module__)

print("\nSpark version:")
print(session.version)

In [0]:
from pyspark.sql import SparkSession

session = SparkSession.builder.getOrCreate()

print("Session created successfully")

test_df = session.createDataFrame(
    [(1, "Dublin"), (2, "Cork")],
    ["id", "location"]
)

test_df.show()

print("DataFrame test completed successfully")

In [0]:
from src.transformations.bronze import build_bronze_dataframe

dimension_ids = [
    "STATISTIC",
    "TLIST(Q1)",
    "C02970V03592",
    "C02969V03591",
    "C03004V03625",
]

dimension_sizes = [1, 2, 1, 2, 1]

dimension_metadata = {
    "STATISTIC": {
        "codes": ["RIQ02"],
        "labels": {"RIQ02": "RTB Average Monthly Rent Report"},
    },
    "TLIST(Q1)": {
        "codes": ["2024Q1", "2024Q2"],
        "labels": {"2024Q1": "2024 Q1", "2024Q2": "2024 Q2"},
    },
    "C02970V03592": {
        "codes": ["-"],
        "labels": {"-": "All bedrooms"},
    },
    "C02969V03591": {
        "codes": ["-", "04"],
        "labels": {"-": "All property types", "04": "Apartment"},
    },
    "C03004V03625": {
        "codes": ["120500"],
        "labels": {"120500": "Dublin"},
    },
}

observations = [1000.0, 1100.0, 1200.0, 1300.0]

test_bronze_df = build_bronze_dataframe(
    spark=spark,
    observations=observations,
    dimension_ids=dimension_ids,
    dimension_sizes=dimension_sizes,
    dimension_metadata=dimension_metadata,
)

test_bronze_df.show(truncate=False)
print("Bronze records:", test_bronze_df.count())

In [0]:
import pytest

result = pytest.main([
    "-v",
    "--tb=long",
    "-p", "no:cacheprovider",
    "--assert=plain",
    str(repo_root / "tests/test_transformations.py::test_bronze_record_count"),
])

In [0]:
from pathlib import Path

for file in repo_root.rglob("*.py"):
    if any(part.startswith(".") for part in file.parts):
        continue

    try:
        lines = file.read_text(encoding="utf-8").splitlines()

        for number, line in enumerate(lines, start=1):
            if "sparkContext" in line:
                print(f"{file.relative_to(repo_root)} — linha {number}")
                print(f"    {line.strip()}")
    except Exception:
        pass

In [0]:
import pytest

result = pytest.main([
    "-v",
    "--tb=long",
    "-p", "no:cacheprovider",
    "--assert=plain",
    str(repo_root / "tests/test_transformations.py::test_bronze_record_count"),
])

print("Pytest exit code:", result)

In [0]:
from pathlib import Path

test_file = repo_root / "tests" / "test_transformations.py"
lines = test_file.read_text(encoding="utf-8").splitlines()

for i, line in enumerate(lines):
    if "test_bronze_record_count" in line:
        start = max(0, i - 3)
        end = min(len(lines), i + 18)

        print("\n".join(
            f"{j + 1}: {lines[j]}"
            for j in range(start, end)
        ))
        break

In [0]:
import pytest

print("Pytest plugins:")

for name, plugin in pytest.config.pluginmanager.list_name_plugin():
    print(name, type(plugin).__module__)

In [0]:
from _pytest.config import get_config

config = get_config()
config.parse([])

print("Loaded plugins:")

for name, plugin in config.pluginmanager.list_name_plugin():
    print(name, type(plugin).__module__)

In [0]:
import pytest
from pyspark.sql import SparkSession

spark_session = SparkSession.builder.getOrCreate()

test_df = spark_session.createDataFrame(
    [(1, "Dublin"), (2, "Cork")],
    ["id", "location"]
)

assert test_df.count() == 2

print("Spark Connect test passed!")

In [0]:
import pytest

def test_spark_connect():
    df = spark.createDataFrame(
        [(1, "Dublin"), (2, "Cork")],
        ["id", "location"]
    )

    assert df.count() == 2

result = pytest.main([
    "-v",
    "-p", "no:cacheprovider",
    "--assert=plain",
    "--tb=short",
    "--override-ini=addopts=",
    "--pyargs", "pytest"
])

print("Pytest exit code:", result)

In [0]:
import pytest

def test_spark_connect():
    df = spark.createDataFrame(
        [(1, "Dublin"), (2, "Cork")],
        ["id", "location"]
    )

    assert df.count() == 2

result = pytest.main([
    "-v",
    "--tb=short",
    "-p", "no:cacheprovider",
    "--assert=plain",
    "--override-ini=addopts=",
    str(repo_root / "tests"),
    "-k", "test_quality_summary",
])

print("Pytest exit code:", result)

In [0]:
import sys

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from tests.conftest import (
    sample_jsonstat_metadata,
    sample_observations,
)

metadata_fixture = sample_jsonstat_metadata.__wrapped__()
observations_fixture = sample_observations.__wrapped__()

print("Metadata loaded:", len(metadata_fixture[0]), "dimensions")
print("Observations:", observations_fixture)

In [0]:
from tests.conftest import bronze_df

bronze_fixture = bronze_df.__wrapped__(
    spark_session=spark,
    sample_jsonstat_metadata=metadata_fixture,
    sample_observations=observations_fixture,
)

bronze_fixture.show(truncate=False)

print("Bronze records:", bronze_fixture.count())

In [0]:
import inspect
from tests.conftest import bronze_df

print(inspect.signature(bronze_df))

In [0]:
from tests.conftest import bronze_df

bronze_fixture = bronze_df.__wrapped__(
    spark=spark,
    sample_jsonstat_metadata=metadata_fixture,
    sample_observations=observations_fixture,
)

bronze_fixture.show(truncate=False)

print("Bronze records:", bronze_fixture.count())

In [0]:
import pytest

fixture_manager = pytest.main

from _pytest.fixtures import FixtureRequest

print("Fixture function:")
print(bronze_df.__wrapped__)

print("\nFixture parameters:")
print(bronze_df.__wrapped__.__code__.co_varnames)

print("\nSpark fixture:")
import tests.conftest as conftest

print(conftest.spark.__wrapped__)
print(conftest.spark.__wrapped__.__code__.co_varnames)

In [0]:
import inspect
from pyspark.sql.connect.session import SparkSession as ConnectSparkSession

print("Spark Connect __getattr__:")
print(inspect.getsource(ConnectSparkSession.__getattr__))

print("\nSpark fixture source:")
print(inspect.getsource(conftest.spark.__wrapped__))

In [0]:
conftest_path = repo_root / "tests" / "conftest.py"

print(conftest_path.read_text(encoding="utf-8"))

In [0]:
import importlib
import tests.conftest as conftest

importlib.reload(conftest)

print("Spark fixture:")
print(conftest.spark_session.__wrapped__.__code__.co_varnames)

print("\nBronze fixture:")
print(conftest.bronze_df.__wrapped__.__code__.co_varnames)

In [0]:
import sys
import types
from pathlib import Path

conftest_path = repo_root / "tests" / "conftest.py"

module = types.ModuleType("tests.conftest")
module.__file__ = str(conftest_path)

sys.modules["tests.conftest"] = module

source = conftest_path.read_text(encoding="utf-8")

exec(compile(source, str(conftest_path), "exec"), module.__dict__)

print("Spark fixture:")
print(module.spark_session.__wrapped__.__code__.co_varnames)

print("\nBronze fixture:")
print(module.bronze_df.__wrapped__.__code__.co_varnames)

In [0]:
import pytest

result = pytest.main([
    "-v",
    "--tb=long",
    "-p", "no:cacheprovider",
    "--assert=plain",
    "--override-ini=addopts=",
    str(repo_root / "tests/test_transformations.py::test_bronze_record_count"),
])

print("Pytest exit code:", result)

In [0]:
import pytest

result = pytest.main([
    "-v",
    "--tb=short",
    "-p", "no:cacheprovider",
    "--assert=plain",
    "--override-ini=addopts=",
    str(repo_root / "tests"),
])

print("Pytest exit code:", result)

In [0]:

import sys
from pathlib import Path

# Locate the repository root
repo_root = Path.cwd().parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pytest

# Run the complete test suite
result = pytest.main([
    "-v",
    "--tb=short",
    "-p", "no:cacheprovider",
    "--assert=plain",
    "--override-ini=addopts=",
    str(repo_root / "tests"),
])

print("\nPytest exit code:", result)

if result == pytest.ExitCode.OK:
    print("All tests passed successfully!")
else:
    print("Some tests failed. Check the output above.")
